# MDP Lesson 4:  Total Reward MDP with two-dimensional state space

This lesson can be downloaded as a notebook, a notebook for colab and a python file [here](https://marmote.gitlabpages.inria.fr/marmote/python_downloads.html)

This C++ notebook is the Xeus-cling counterpart of the Python lesson. It keeps the same modelling steps and the same pedagogical order while relying on the official Marmote C++ API illustrated in `xpl/exampleMDP31/exampleMDP31.cpp`.

### Using the library

In [ ]:
// --- Marmote configuration for Xeus-cling ---
// These directives load the Marmote include paths and shared libraries.
#ifdef _WIN32
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteCore")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteMDP")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteMarkovChain")
#pragma cling add_library_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/bin")
#pragma cling load("marmoteCore.dll")
#pragma cling load("marmoteMarkovChain.dll")
#pragma cling load("marmoteMDP.dll")
#else
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteCore")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteMDP")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteMarkovChain")
#pragma cling add_library_path("/home/assia/miniconda3/envs/xeus-cpp-env/lib")
#pragma cling load("libmarmoteCore.so")
#pragma cling load("libmarmoteMarkovChain.so")
#pragma cling load("libmarmoteMDP.so")
#endif



In [ ]:
// --- Standard C++ utilities used in the notebook ---
// These headers support formatted output and basic containers.
#include <iostream>
#include <string>

// --- Marmote headers used in this lesson ---
// They provide the C++ counterparts of the Python Marmote objects.
#include "marmoteCore/marmoteBox.h"
#include "marmoteCore/marmoteFullMatrix.h"
#include "marmoteCore/marmoteInterval.h"
#include "marmoteCore/marmoteSparseMatrix.h"
#include "marmoteMDP/marmoteFeedbackSolutionMDP.h"
#include "marmoteMDP/marmoteTotalRewardMDP.h"

// --- Convenience declarations for the notebook cells below ---
// They simplify the pedagogical examples without changing the model.
using namespace std;


## Build a MDP associated with a Stochastic Shortest Path

### Description of the model

We consider the (classic) four rooms model which is represented on the figure below. This model is related to a *Stochastic Shortest Path*. The state space is divided into 4 rooms, each with (5 x 5) positions. You can only move from one room to another at particular locations. There is also an exit state from the system, which yields a gain. Moving from one position to another incurs a cost.

<img src="../_images/FourRooms.png" alt="picture of the MDP" width="350">

More Precisely:  

We assume that the state space has two dimensions 11 lines x 10 columns ([0,10]*[0,9])  

If we reach the state (9,2) we receive a reward of -1 and we go in state (10,2). In any state in line 10 we stay in this state without receiving anything (absorbing states).
 
There is wall between line 4 and 5 that can be crossed   
 -> at states (4,2)->(5,2) and (4,7)->(5,7)    
 -> at states (5,2)->(4,2) and (5,7)->(4,7)  

There is a wall between column 4 and 5 that can be crossed   
 -> at states (2,4)->(2,5) and (7,4)->(7,5)  
 -> at states (2,5)->(2,4) and (7,5)->(7,4)  
   
There are 4 actions: 0 is up 1 is down 2 is left and 3 is right. With a probability *p=0.9* the action has the desired effect and with probability *1-p* the action has no effect. 
Performing an action in any state except (9,2) has a cost of 1.

## Multidimensional State Space

In all what follows we consider a two dimensionals state space with first dimension equals to 11 and second dimension equal to 10.

### Creating Spaces

**Definitions of the state**

The object used to create the state space is a `MarmoteBox` with dimension 11 x 10. 
First define the dimensions of the box in an array.

In [ ]:
// Define the sizes of the two dimensions of the box.
stateType dims[2] = {11, 10};

Then create the box (and illustrate this by printing the cardinal, the dimension and the object).

In [ ]:
// Create the two-dimensional state space and display its main features.
MarmoteBox *stateSpace = new MarmoteBox(2, dims);

stateType dim_SS = stateSpace->Cardinal();
cout << "State Space cardinal " << dim_SS << endl;
cout << "State Space dimension " << stateSpace->tot_nb_dims() << endl;
cout << "State Space type " << stateSpace->toString() << endl;

Create the action space as an interval between 0 and 3.

In [ ]:
// Create the action space with the four admissible moves.
MarmoteInterval *actionSpace = new MarmoteInterval(0, 3);
stateType dim_AS = actionSpace->Cardinal();

**Associate a state to a variable**

It is possible to associate a state either to a buffer or to an array in Python. In C++, the direct equivalent is a `MarmoteState` buffer allocated by the state space. The buffer stores the different values of a given state.

Furthermore to each state is associated an index. It is then possible to pass from a state to an index and conversely.

Here one creates two buffers: one to represent the initial state (before transition) and the second one to represent the state after transition. The state is *(0,0)* in the following instruction.

In [ ]:
// First buffer allows us to manage the initial state.
MarmoteState etat = stateSpace->StateBuffer();
// Second buffer allows us to manage the final state (after transition).
MarmoteState sortie = stateSpace->StateBuffer();

etat[0] = 0;
etat[1] = 0;
sortie[0] = 0;
sortie[1] = 0;

stateType k = 0;
stateType indexO = 0;
stateType indexD = 0;
int l = 0;
int c = 0;

**Filling Cost matrix**

We fill in the matrix: all the costs are equal to 1 except in 9,2 in which it is equal to -1 and in line 10 in which it is equal to 0.

In [ ]:
cout << "Fill in Cost Matrix" << endl;
FullMatrix *CostMat = new FullMatrix(dim_SS, dim_AS);

stateSpace->FirstState(etat);
for (k = 0; k < dim_SS; k++) {
    // Compute the index of the state.
    indexO = stateSpace->Index(etat);
    // For each state we give a value to every action.
    for (stateType a = 0; a < dim_AS; a++) {
        CostMat->setEntry(indexO, a, 1.0);
    }
    stateSpace->NextState(etat);
}

// Replace the term in (9,2) for action UP by -1.
etat[0] = 9;
etat[1] = 2;
indexO = stateSpace->Index(etat);
cout << "index of state (9,2) " << indexO << endl << endl;
CostMat->setEntry(indexO, 0, -1.0);

// Fill line 10: all costs are equal to zero.
etat[0] = 10;
for (k = 0; k < 10; k++) {
    etat[1] = k;
    indexO = stateSpace->Index(etat);
    CostMat->setEntry(indexO, 0, 0.0);
    CostMat->setEntry(indexO, 1, 0.0);
    CostMat->setEntry(indexO, 2, 0.0);
    CostMat->setEntry(indexO, 3, 0.0);
}

### Use a third way to build the MDP

In what follows, we use a constructor that does not require to build the list of matrices. They are added one by one.  
Please notice that the name of the matrices should differ.

In [ ]:
// Set the optimisation criterion and build the total reward MDP.
string criterion = "min";

cout << "Begining of building MDP" << endl;
TotalRewardMDP *mdpSSP = new TotalRewardMDP(criterion, stateSpace, actionSpace, CostMat);
cout << "End of building MDP" << endl;

Then we fill in the transition matrices. Then we have four matrices to fill in.

First we complete matrix for action 0 (UP). All the states are browsed by iterating over the rows and columns. For each new state value, the index is calculated, then the possible output states and their indexes are computed to fill in the entry.

In [ ]:
cout << "Add matrices" << endl;

double p = 0.9;

SparseMatrix *P0 = new SparseMatrix(dim_SS);
for (l = 0; l < 10; l++) {
    for (c = 0; c < 10; c++) {
        // Define a state and get its index.
        etat[0] = l;
        etat[1] = c;
        indexO = stateSpace->Index(etat);
        if ((l == 4) || (l == 9)) {
            if ((l == 4) && ((c == 2) || (c == 7))) {
                // I am on a door: either I move up with probability p, or I stay.
                sortie[0] = l + 1;
                sortie[1] = c;
                indexD = stateSpace->Index(sortie);
                P0->setEntry(indexO, indexD, p);
                P0->setEntry(indexO, indexO, 1 - p);
            } else {
                if ((l == 9) && (c == 2)) {
                    // Special door leading to the absorbing line.
                    sortie[0] = l + 1;
                    sortie[1] = c;
                    indexD = stateSpace->Index(sortie);
                    P0->setEntry(indexO, indexD, p);
                    P0->setEntry(indexO, indexO, 1 - p);
                } else {
                    // I am on the wall l=4 or l=9: I stay in the same state.
                    P0->setEntry(indexO, indexO, 1.0);
                }
            }
        } else {
            // I am in a room: either I move up with probability p, or I stay.
            sortie[0] = l + 1;
            sortie[1] = c;
            indexD = stateSpace->Index(sortie);
            P0->setEntry(indexO, indexD, p);
            P0->setEntry(indexO, indexO, 1 - p);
        }
    }
}

// Fill in the last line.
for (c = 0; c < 10; c++) {
    etat[0] = 10;
    etat[1] = c;
    indexO = stateSpace->Index(etat);
    P0->setEntry(indexO, indexO, 1.0);
}

mdpSSP->AddMatrix(0, P0);
cout << "Added matrix (action 0)" << endl;

We complete the remaining matrices.

In [ ]:
// Complete matrix for action 1 (DOWN).
SparseMatrix *P1 = new SparseMatrix(dim_SS);

for (l = 0; l < 10; l++) {
    for (c = 0; c < 10; c++) {
        etat[0] = l;
        etat[1] = c;
        indexO = stateSpace->Index(etat);
        if ((l == 5) || (l == 0)) {
            if ((l == 5) && ((c == 2) || (c == 7))) {
                // I am on a door: either I move down with probability p, or I stay.
                sortie[0] = l - 1;
                sortie[1] = c;
                indexD = stateSpace->Index(sortie);
                P1->setEntry(indexO, indexD, p);
                P1->setEntry(indexO, indexO, 1 - p);
            } else {
                // I am on the wall l=5 or l=0: I stay in the same state.
                P1->setEntry(indexO, indexO, 1.0);
            }
        } else {
            // I am in a room: either I move down with probability p, or I stay.
            sortie[0] = l - 1;
            sortie[1] = c;
            indexD = stateSpace->Index(sortie);
            P1->setEntry(indexO, indexD, p);
            P1->setEntry(indexO, indexO, 1 - p);
        }
    }
}

// Fill in the last line.
for (c = 0; c < 10; c++) {
    etat[0] = 10;
    etat[1] = c;
    indexO = stateSpace->Index(etat);
    P1->setEntry(indexO, indexO, 1.0);
}

mdpSSP->AddMatrix(1, P1);
cout << "Added matrix (action 1)" << endl;

// Define matrix for action 2 (LEFT).
SparseMatrix *P2 = new SparseMatrix(dim_SS);
for (l = 0; l < 10; l++) {
    for (c = 0; c < 10; c++) {
        etat[0] = l;
        etat[1] = c;
        indexO = stateSpace->Index(etat);
        if ((c == 5) || (c == 0)) {
            if ((c == 5) && ((l == 2) || (l == 7))) {
                // I am on a door: either I move left with probability p, or I stay.
                sortie[0] = l;
                sortie[1] = c - 1;
                indexD = stateSpace->Index(sortie);
                P2->setEntry(indexO, indexD, p);
                P2->setEntry(indexO, indexO, 1 - p);
            } else {
                // I am on the wall c=5 or c=0: I stay in the same state.
                P2->setEntry(indexO, indexO, 1.0);
            }
        } else {
            // I am in a room: either I move left with probability p, or I stay.
            sortie[0] = l;
            sortie[1] = c - 1;
            indexD = stateSpace->Index(sortie);
            P2->setEntry(indexO, indexD, p);
            P2->setEntry(indexO, indexO, 1 - p);
        }
    }
}

// Fill in the last line.
for (c = 0; c < 10; c++) {
    etat[0] = 10;
    etat[1] = c;
    indexO = stateSpace->Index(etat);
    P2->setEntry(indexO, indexO, 1.0);
}

mdpSSP->AddMatrix(2, P2);
cout << "Added matrix (action 2)" << endl;

// Define matrix for action 3 (RIGHT).
SparseMatrix *P3 = new SparseMatrix(dim_SS);
for (l = 0; l < 10; l++) {
    for (c = 0; c < 10; c++) {
        etat[0] = l;
        etat[1] = c;
        indexO = stateSpace->Index(etat);
        if ((c == 4) || (c == 9)) {
            if ((c == 4) && ((l == 2) || (l == 7))) {
                // I am on a door: either I move right with probability p, or I stay.
                sortie[0] = l;
                sortie[1] = c + 1;
                indexD = stateSpace->Index(sortie);
                P3->setEntry(indexO, indexD, p);
                P3->setEntry(indexO, indexO, 1 - p);
            } else {
                // I am on the wall c=4 or c=9: I stay in the same state.
                P3->setEntry(indexO, indexO, 1.0);
            }
        } else {
            // I am in a room: either I move right with probability p, or I stay.
            sortie[0] = l;
            sortie[1] = c + 1;
            indexD = stateSpace->Index(sortie);
            P3->setEntry(indexO, indexD, p);
            P3->setEntry(indexO, indexO, 1 - p);
        }
    }
}

// Fill in the last line.
for (c = 0; c < 10; c++) {
    etat[0] = 10;
    etat[1] = c;
    indexO = stateSpace->Index(etat);
    P3->setEntry(indexO, indexO, 1.0);
}

mdpSSP->AddMatrix(3, P3);
cout << "Added matrix (action 3)" << endl;

cout << "Finishing Adding matrices MDP" << endl;
cout << "Writing MDP" << endl;
cout << *mdpSSP << endl;

## Resolution of the MDP

In the Python notebook the solution object is displayed with `print(optimum2)`. In C++, the robust equivalent in Xeus-cling is to call `Write()` on the `FeedbackSolutionMDP` object.

In [ ]:
// Set the solver parameters and compute the optimal policy by value iteration.
double epsilon = 0.0001;
int maxIter = 250;

cout << "\nPrinting solution from value iteration" << endl;
// Marmote version note: the notebook uses the feedback-solution type returned
// by the current API, then displays it with Write().
FeedbackSolutionMDP *optimum2 = mdpSSP->ValueIteration(epsilon, maxIter);
optimum2->Write();

## Analysis of the results

### Print policy dimension by dimension

A `FeedbackSolutionMDP` can be printed dimension by dimension with method `SolutionByDim` whose first parameter is the dimension to be scanned. Below we scan the columns. The line is fixed and we let vary the columns.

The C++ API also provides `WriteSolutionByDim`, but `SolutionByDim` is closer to the Python notebook because it returns a string that we can display explicitly.

In [ ]:
// Display the policy line by line by scanning the second dimension.
// Marmote version note: this notebook uses SolutionByDim(), which returns a
// printable string; it avoids the obsolete/fragile writeSolution() path.
cout << "Print solution by dimension (line by line)" << endl;
string line = optimum2->SolutionByDim(1, stateSpace);
cout << line << endl;

### Enumerating the policy

We can scan the policy by the way of the *iterator* of space. We also create a *buffer* to store the state.

In [ ]:
// Create the buffer.
MarmoteState bbuf = stateSpace->StateBuffer();
cout << "Printing State Space Path and value function with a browsing by iterating space" << endl;

// Initial state: bbuf receives the value of the first state of the state space.
stateSpace->FirstState(bbuf);

// Scan the whole state space.
for (k = 0; k < stateSpace->Cardinal(); k++) {
    indexO = stateSpace->Index(bbuf);
    l = bbuf[0];
    c = bbuf[1];

    cout << "--State in line=" << l << " column=" << c;
    if ((c <= 4) && (l <= 4)) {
        cout << " --in Room at Bottom Left  ";
    }
    if ((c <= 4) && (l >= 5)) {
        cout << " --in Room at Top Left     ";
    }
    if ((c >= 5) && (l <= 4)) {
        cout << " --in Room at Bottom Right ";
    }
    if ((c >= 5) && (l >= 5)) {
        cout << " --in Room at Top Right    ";
    }

    cout << " --Optimal Action=" << optimum2->getActionIndex(indexO)
         << " --Value=" << optimum2->getValueIndex(indexO) << endl;

    stateSpace->NextState(bbuf);
}

In [ ]:
// Release the objects allocated in this notebook.
delete optimum2;
delete mdpSSP;
delete stateSpace;
delete actionSpace;
delete[] etat;
delete[] sortie;
delete[] bbuf;